In [ ]:
import sys
import os
from util import Score_calculate as score
from transformers import T5Tokenizer, T5ForConditionalGeneration
import pandas as pd
from tqdm import tqdm
import torch

In [ ]:
# Define generation function
def generate_smiles_from_caption(caption, tokenizer, model, max_length=128):
    inputs = tokenizer(caption, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.strip()

In [ ]:
# SingMolT5 Model Generation and metrics computation

# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained(R"YOUR_MODEL_PATH/SingMolT5_large", model_max_length=128)
model = T5ForConditionalGeneration.from_pretrained(R'YOUR_MODEL_PATH/SingMolT5_large')

# Load testing dataset
df = pd.read_csv("Dataset/MolGen_Test.csv")
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

# Set cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Generate SMILES strings
ft_generated_smiles = []
for caption in tqdm(captions, desc="Generating SMILES"):
    pred = generate_smiles_from_caption(caption, tokenizer, model)
    ft_generated_smiles.append(pred)

# Compute evaluation metrics
with score.suppress_rdkit_warnings():
    results = score.evaluate_batch(true_smiles, ft_generated_smiles)

score.result_output(results)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Generating SMILES: 100%|██████████| 60/60 [00:43<00:00,  1.38it/s]


=== Evaluation Results (Average over Batch) ===
BLEU                  : 0.5804
Levenshtein Similarity: 0.6529
MACCS FTS             : 0.7851
RDK FTS               : 0.5280
Morgan FTS            : 0.4885
Validity              : 0.9833


In [ ]:
# MolT5 Model Generation and metrics computation

# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained(R"YOUR_MODEL_PATH/MolT5large_Cap2Sm", model_max_length=128)
model = T5ForConditionalGeneration.from_pretrained(R'YOUR_MODEL_PATH/MolT5large_Cap2Sm')

# Load testing dataset
df = pd.read_csv("Dataset/MolGen_Test.csv")
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

# Set cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Generate SMILES strings
mt5_generated_smiles = []
for caption in tqdm(captions, desc="Generating SMILES"):
    pred = generate_smiles_from_caption(caption, tokenizer, model)
    mt5_generated_smiles.append(pred)

# Compute evaluation metrics
with score.suppress_rdkit_warnings():
    results = score.evaluate_batch(true_smiles, mt5_generated_smiles)

score.result_output(results)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/anaconda3/envs/zyhEnv/lib/python3.8/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Generat


=== Evaluation Results (Average over Batch) ===
BLEU                  : 0.4924
Levenshtein Similarity: 0.5717
MACCS FTS             : 0.5768
RDK FTS               : 0.3136
Morgan FTS            : 0.2512
Validity              : 0.9833


In [ ]:
# T5 Model Generation and metrics computation

# Load model and tokenizer
tokenizer = T5Tokenizer.from_pretrained(R"YOUR_MODEL_PATH/T5large_Cap2Sm", model_max_length=128)
model = T5ForConditionalGeneration.from_pretrained(R'YOUR_MODEL_PATH/T5large_Cap2Sm')

# Load testing dataset
df = pd.read_csv("Dataset/MolGen_Test.csv")
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

# Set cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# Generate SMILES strings
t5_generated_smiles = []
for caption in tqdm(captions, desc="Generating SMILES"):
    pred = generate_smiles_from_caption(caption, tokenizer, model)
    t5_generated_smiles.append(pred)

# Compute evaluation metrics
with score.suppress_rdkit_warnings():
    results = score.evaluate_batch(true_smiles, t5_generated_smiles)

score.result_output(results)

Generating SMILES: 100%|██████████| 60/60 [00:45<00:00,  1.32it/s]


=== Evaluation Results (Average over Batch) ===
BLEU                  : 0.4641
Levenshtein Similarity: 0.5713
MACCS FTS             : 0.5632
RDK FTS               : 0.3051
Morgan FTS            : 0.2106
Validity              : 0.9833


In [8]:
print(f"Model is on device: {model.device}")

Model is on device: cuda:0


In [9]:
print(ft_generated_smiles)

['C1=CC=NC(=C1)CC#CC2=CC=CC=N2', 'CC(=O)C1=CC=NC=C1', 'C1=CC=NC(=C1)CC#CC2=CC=CC=N2', 'C1=CC(=CC=C1C#CC2=CC=C(C=C2)C(=O)O)C(=O)O', 'C1=CC(=CC=C1C#CC2=CC=C(C=C2)C(=O)O)C(=O)O', 'CC#CC1=CC(=C(C=C1)C#CC2=CC=C(C=C2)C(=O)O)C(=O)O', 'C1=CC2=C(C=C1)C3=C(C2=O)C=CC(=C3)S', 'C1=CC2=C3C(=C1)C=CC4=CC(=CC(=C43)S2)S', 'C1=CC2=C3C(=C1)C=CC4=CC(=CC(=C43)C=C2)S', 'C1=CC(=CC=C1C2=CC=C(C=C2)N)N', 'C1=CC(=CC=C1C2=CC=C(C=C2)N)N', 'C1=CC(=CC=C1C2=CC=C(C=C2)N)N', 'CSC1=CC=C2C=C(SC)C=CC2=C1', 'CSC1=CC2=C(C=CC(=C2)SC)C=C1', 'CSC1=CC=C2C=C(SC)C=CC2=C1', 'C1=CC(=NC(=C1)C#N)S', 'C1=CN=CC=C1S', 'C1=CN=CC=C1S', 'CC(=O)SC1=CC=NC(=C1)CSC(=O)C', 'CC(=O)SC1=CC=NC(=C1)CSC', 'CC(=O)SC1=CC=NC=C1', 'CNC1=CC=C(C#N)C#N', 'CNC1=CC=C(C=C1)C#N', 'CNC1=CC=C(C#N)C=C1', 'C1=CC(=CC=C1C#N)C#N', 'C1=CC(=CC=C1C#N)C#N', 'C1=CC(=CC=C1C#N)C#N', 'C1=CC2=C(C=CC=C2I)C=C1', 'C1=CC2=C(C=CC=C2I)C=C1I', 'C1=CC2=C(C=CC=C2I)C=C1I', 'C1=CC2=C(C=C1SC)C3=C(C=C2)C=C(S3)C4=CC=CS4', 'C1=CSC(=C1)C2=CC=C(S2)C3=CC=C(S3)C4=CC=CS4', 'C1=CC2=C(C=C1SC)C3=C(C=

In [10]:
print(mt5_generated_smiles)

['C1=CC=NC(=C1)CC#N', 'CC(=O)N1CCCC1', 'C1=CC=NC(=C1)CC2=CC=CC=N2', 'C1=CC=C(C(=C1)C2=CC=CC=C2C(=O)O)C(=O)O', 'C1=CC=C(C(=C1)C2=CC=CC=C2C(=O)O)C(=O)O', 'CCC1=CC=CC(=C1N(CC(=O)OCC)CC(=O)OCC)C2=CC=CC=C2', 'C1=CC=C2C(=C1)C=C3C=CC(=CC3=C2)S(=O)(=O)[O-]', 'C1=CC=C2C(=C1)C=C3C=CC=CC3=N2', 'C1=CC=C2C(=C1)C=C3C=CC4=CC=CC=C4C3=CC=C2S(=O)(=O)O', 'C1=CC=C(C=C1)C2=CC=CC=C2N', 'C1=CC=C(C=C1)C2=CC=CC=C2N', 'C1=CC=C(C=C1)C2=CC=CC=C2N', 'CC1=C(C2=CC=CC=C2C(=C1)SC)C', 'CN=C1C=CC2=CC=CC=C21', 'CC1=C(C2=CC=CC=C2C(=C1)C)SC', 'C1=C(C=[N+](C=C1S(=O)(=O)N)[S-])[S-]', 'C1=CC=NC(=C1)S(=O)(=O)N', 'C1=CNC(=S)N=C1', 'C[C@@H]1[C@H](C=C[C@@H](O1)N2C=CC(=NC2=O)C)SC', 'C[S+](C)C1=CN=CC=C1', 'C[S+](C)C1=CC=CC=C1', 'CNC1=CC=CC=C1', 'CN(C)C1=CC=C(C=C1)C2=CC=CC=N2', 'CNC1=CC=CC=C1', 'C1=CC=C(C=C1)C#N', 'C1=CC=C(C=C1)C#N', 'C1=CC=C(C=C1)C#N', 'C1=CC=C2C(=C1)C=CC3=C2C4=CC=CC=C4C=C3', 'C1=CC=C2C(=C1)C=CC3=C2C4=CC=CC=C4C=C3', 'C1=CC=C2C(=C1)C=CC3=C2C4=CC=CC=C4C=C3', 'C1=CC=C2C(=C1)C3=CC=CC=C3S2', 'C1=CC=C2C(=C1)C3=CC=CC=C3S2

In [11]:
print(t5_generated_smiles)

['C1=CC(=CN=C1)C2=CC=NC=C2', 'CC(=O)N1CCCC1', 'CC(=O)C1=CC=CN1', 'C1=CC=C(C=C1)C(C2=CC=CC=C2)(C3=CC=CC=C3)C(=O)O', 'C1=CC=C(C=C1)C(C2=CC=CC=C2)C(=O)[O-]', 'CCC1=CC=CC=C1CC2=CC=CC=C2', 'C1=CC=C2C(=C1)C=CC3=CC=CC=C3S2', 'CC1=CC2=C(C=C1S(=O)(=O)[O-])C(=C3C=C(C(=[NH+]CC)C=C3O2)C)C4=C(C=C(C=C4)S(=O)(=O)[O-])S(=O)(=O)[O-]', 'CC1=C(C2=CC3=C(C(=C(C=C3)S(=O)(=O)O)S(=O)(=O)O)C(=C2C1=O)O)C', 'C1=CC=C(C=C1)C2=CC=CC=C2N', 'C1=CC=C(C=C1)C2=CC=CC=C2N', 'C1=CC=C(C=C1)C2=CC=CC=C2N', 'CSC1=CC=CC2=CC=CC=C21', 'CSC1=CC=CC2=CC=CC=C21', 'CSC1=CC=CC2=CC=CC=C21', 'C1=CC(=NC(=C1)S(=O)(=O)O)S(=O)(=O)O', 'C1=CC=NC(=C1)S(=O)(=O)[O-]', 'C1=CC=NC(=C1)SCC2=CC=CC=N2', 'CSCC[C@H]1C(=O)NC(=C)C1', 'CC(=O)SCC1=CN=C(N=C1)C2=CC=CC=N2', 'CSCC[C@@H]1[C@H](C(=O)N=C(S1)N)NC(=O)[C@@H](C(=O)O)N', 'C[N+]1=CC=CC=C1/C=C/C2=CC=CC=C2', 'CN(C)C1=CC=C(C=C1)C2=CC=CC=C2', 'C[N+]1=CC=CC=C1/C=C/C2=CC=CC=C2', 'C1=CC=C(C=C1)C(=O)C2=CC=C(C=C2)N=C=O', 'C1=CC=C2C(=C1)C=CC(=C2N=C=O)C#N', 'C1=CC=C(C=C1)C#N', 'C1=CC=C2C(=C1)C=C(C(=C2I)N=I)I', 'C1=